In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
net=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
X=torch.rand(2,20)
net(X)

tensor([[ 0.0614,  0.1106,  0.0116,  0.2093,  0.0128,  0.0920, -0.2041, -0.0447,
         -0.0581,  0.1849],
        [ 0.1302,  0.1378,  0.1002,  0.1865,  0.0312,  0.1202, -0.1390, -0.0360,
         -0.0846,  0.2861]], grad_fn=<AddmmBackward0>)

In [3]:
class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.hidden=nn.Linear(20,256)
    self.out=nn.Linear(256,10)
  def forward(self,X):
    return self.out(F.relu(self.hidden(X)))
  
net=MLP()
net(X)

tensor([[ 1.7812e-01, -2.0043e-04,  7.1726e-02,  1.1923e-01,  1.4078e-01,
          1.1610e-02, -5.1644e-02,  1.7277e-01,  3.0890e-01, -1.7654e-01],
        [ 2.1366e-01,  1.7390e-01,  6.6693e-02,  1.6187e-01,  1.1993e-01,
          1.0194e-01, -1.4353e-01,  2.6867e-02,  3.1247e-01, -2.2065e-01]],
       grad_fn=<AddmmBackward0>)

In [4]:
class MySequential(nn.Module):
  def __init__(self,*args):
    super().__init__()
    for idx,module in enumerate(args):
      self._modules[str(idx)]=module
      
  def forward(self,x):
    for module in self._modules.values():
      x=module(x)
    return x

In [5]:
class FixedHiddenMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.rand_weight=torch.rand((20,20),requires_grad=True)
    self.linear=nn.Linear(20,20)

  def forward(self, X):
    X=self.linear(X)
    X=F.relu(torch.mm(X,self.rand_weight)+1)
    X=self.linear(X)
    while X.abs().sum()>1:
      X/=2
    return X.sum()

In [6]:
net=FixedHiddenMLP()
net(X)

tensor(-0.0323, grad_fn=<SumBackward0>)

In [11]:
class NestMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.net=nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
    self.linear=nn.Linear(32,16)
    
  def forward(self,X):
    return self.linear(self.net(X))

chimera=nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())
chimera(X)

tensor(0.1141, grad_fn=<SumBackward0>)